<a href="https://colab.research.google.com/github/VedashreeMirajkar/flyrank-machine-learning/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
%pip -q install duckdb huggingface_hub pandas numpy

In [17]:
import os
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found in Colab Secrets.")

# Connect DuckDB
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Store token safely in DuckDB
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute("""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN getvariable('hf_token'))
""")

# Dataset paths
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

# February = feature window
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

# March = label window
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

# Dimension tables
DIM = f"read_parquet('{REL}/dim_content.parquet')"
CLI = f"read_parquet('{REL}/dim_clients.parquet')"

print("✅ Warehouse connected")
print("Feature window: February 2026")
print("Label window: March 2026")

✅ Warehouse connected
Feature window: February 2026
Label window: March 2026


In [18]:
schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM {FEB}
""").df()

schema[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis**

One final analysis row represents **one content item for one client** (`client_hash_id × content_hash_id`).

The source table `fact_content_daily_performance` is daily data, with one row representing a client × content × day.

We use two separate time windows:

| Window | Dates | Purpose |
|---|---|---|
| February 2026 | 2026-02-01 to 2026-02-28 | Feature window |
| March 2026 | 2026-03-01 to 2026-03-31 | Label window |

February is used for information that would have been available by **2026-02-28**.

March is used only for the outcome we want to predict.

The two windows do not overlap, which helps prevent future information from leaking into the features.

**Universe**

A content item enters the analysis when:

1. It has at least 100 GSC impressions in February.
2. It has at least 3 GSC clicks in February.
3. It is published.
4. It existed by 2026-02-28.

**Label**

`went_dark = 1` when the content item has zero GSC clicks during March 2026. Otherwise, `went_dark = 0`.

The purpose is to identify pages that are worth investigating for possible performance decline.

In [19]:
# Check the date range and raw grain of the February feature window.

windows = con.sql(f"""
    SELECT
        'February 2026' AS window_name,
        MIN(report_date) AS first_day,
        MAX(report_date) AS last_day,
        COUNT(DISTINCT report_date) AS n_days,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_contents,
        COUNT(*) AS n_rows
    FROM {FEB}

    UNION ALL

    SELECT
        'March 2026' AS window_name,
        MIN(report_date) AS first_day,
        MAX(report_date) AS last_day,
        COUNT(DISTINCT report_date) AS n_days,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_contents,
        COUNT(*) AS n_rows
    FROM {MAR}
""").df()

print(windows.to_string(index=False))

# Check whether the raw daily grain has duplicate
# report_date × client × content combinations.

duplicates = con.sql(f"""
    SELECT COUNT(*)
    FROM (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id
        FROM {FEB}
        GROUP BY
            report_date,
            client_hash_id,
            content_hash_id
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]

print()
print(f"Duplicate daily grain groups: {duplicates}")

assert duplicates == 0

  window_name  first_day   last_day  n_days  n_clients  n_contents  n_rows
February 2026 2026-02-01 2026-02-28      28         54      321546 7355108
   March 2026 2026-03-01 2026-03-31      31         55      331437 9841378

Duplicate daily grain groups: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

The following February 2026 values can be used as features because they are available before the March label window:

- `gsc_impressions` — total February impressions
- `gsc_clicks` — total February clicks
- `ctr_feb` — February clicks divided by February impressions
- `pos_feb` — February impression-weighted average position
- `days_with_imps_feb` — number of February days with impressions

These features are available by the end of February 2026.

### Label

The March outcome is:

- `went_dark = 1` if March GSC clicks are zero
- `went_dark = 0` otherwise

The March clicks are used only to create the label and are not used as model features.

### Context

These fields are used for filtering, joining, grouping, or reporting:

- `client_hash_id`
- `content_hash_id`
- `report_date`
- `gsc_data_available`
- `is_published`
- `is_deleted`

These are not model features.

### Excluded

The following information is excluded from the model features:

- March `gsc_clicks` — it is used to create the label and therefore would cause leakage.
- March `gsc_impressions` — it belongs to the future label window.
- `trend_direction` — it is directly related to the target and can leak the label.
- `trend_pct` — it is also derived from future performance and can leak the label.
- `last_optimized_date` — this contains information that was not available at the February decision point.
- `optimization_eligible_date` — this is future information.
- `content_updated_date` — the available snapshot can contain information after the February cutoff.

The main rule is: a feature must contain information that could have been known on 2026-02-28.

In [20]:
# Build the February universe.
# Only February information is used here.

universe = con.sql(f"""
    WITH feb_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS imp_feb,
            SUM(gsc_clicks) AS clk_feb
        FROM {FEB}
        WHERE gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id
        HAVING
            SUM(gsc_impressions) >= 100
            AND SUM(gsc_clicks) >= 3
    )

    SELECT
        f.*
    FROM feb_agg f

    JOIN {DIM} d
        USING (client_hash_id, content_hash_id)

    WHERE d.is_published IS TRUE
      AND d.content_created_date <= DATE '2026-02-28'
""").df()

print(f"Universe rows: {len(universe):,}")
print(f"Clients: {universe['client_hash_id'].nunique():,}")
print()
print(universe[["imp_feb", "clk_feb"]].describe().to_string())

Universe rows: 29,700
Clients: 31

             imp_feb       clk_feb
count   29700.000000  29700.000000
mean     4878.206195     18.595084
std      7771.160910     46.995007
min       100.000000      3.000000
25%      1322.750000      4.000000
50%      2561.000000      8.000000
75%      5370.000000     17.000000
max    167303.000000   3310.000000


In [21]:
# Create the March label.
# March is the future outcome and is NOT used as a feature.

march_label = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_mar,
        SUM(gsc_clicks) AS clk_mar
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

# Join March outcome to the February universe.
frame = universe.merge(
    march_label,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# No March record means no observed March traffic.
frame[["imp_mar", "clk_mar"]] = frame[["imp_mar", "clk_mar"]].fillna(0)

# Target:
# 1 = zero March clicks
# 0 = at least one March click
frame["went_dark"] = (frame["clk_mar"] == 0).astype(int)

print(f"Final rows: {len(frame):,}")
print(f"Positive labels: {frame['went_dark'].sum():,}")
print(f"Base rate: {frame['went_dark'].mean():.3f}")

Final rows: 29,700
Positive labels: 1,506
Base rate: 0.051


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [22]:
# Verification 1: final analysis grain should be
# one client × one content item.

duplicate_keys = frame.duplicated(
    subset=["client_hash_id", "content_hash_id"]
).sum()

print(
    f"Duplicate client × content keys: {duplicate_keys}"
)

assert duplicate_keys == 0

Duplicate client × content keys: 0


In [23]:
# Verification 2: verify the final universe size
# and the February/March date windows.

print(f"Final universe rows: {len(frame):,}")

print()
print("February:")
print(
    f"2026-02-01 to 2026-02-28"
)

print()
print("March:")
print(
    f"2026-03-01 to 2026-03-31"
)

print()
print(
    "February feature rows:",
    len(universe)
)

print(
    "Final rows after March left join:",
    len(frame)
)

assert len(frame) == len(universe)

Final universe rows: 29,700

February:
2026-02-01 to 2026-02-28

March:
2026-03-01 to 2026-03-31

February feature rows: 29700
Final rows after March left join: 29700


In [24]:
# Verification 3: check the GSC availability flag.
# IS TRUE is intentional: unavailable data must not be treated as real zero traffic.

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS rows_with_gsc_available
    FROM {FEB}
""").df()

print(availability_check.to_string(index=False))

 total_rows  rows_with_gsc_available
    7355108                  2621783


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limitations

**1. Uneven history**

Not every client has the same amount of February data. Therefore, some February feature totals are based on fewer observed days than others.

**2. Client concentration**

The data is concentrated across clients, so a random row-level train/test split could make performance look better than it really is. Future model evaluation should split by client rather than randomly splitting individual rows.

**3. GSC availability**

Some rows may have unavailable GSC data. An unavailable value should not automatically be interpreted as zero traffic.

**4. Snapshot limitation**

Some content attributes come from a later dimension snapshot rather than a historical February snapshot. Therefore, those attributes may not perfectly represent the content state at the February decision point.

**5. The label does not explain the cause**

`went_dark` only identifies that March clicks reached zero. It does not explain why this happened. Possible reasons could include search demand changes, SERP changes, content changes, or other external factors.

Therefore, this analysis should be treated as **decision-support and triage**, not proof of causation.

In [25]:
# Simple limitation check: how many days of February
# are available for each client?

client_days = con.sql(f"""
    SELECT
        client_hash_id,
        COUNT(DISTINCT report_date) AS days_present
    FROM {FEB}
    GROUP BY client_hash_id
    ORDER BY days_present
""").df()

print("Minimum February days:", client_days["days_present"].min())
print("Maximum February days:", client_days["days_present"].max())

print()
print(
    "Clients with fewer than 28 February days:",
    (client_days["days_present"] < 28).sum()
)

Minimum February days: 3
Maximum February days: 28

Clients with fewer than 28 February days: 16


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.